## PHC TXDOT – Storage & Widget Setup

### Required Storage
- **Type:** Azure Data Lake Storage Gen2 (ADLS Gen2)
- **Storage Account:** `stphctxdotdev`
- **Container:** `unity`
- **Access:** Databricks Access Connector (managed identity) must have **Storage Blob Data Contributor**

---

### Widget Values Examples Only ###

#### catalog_managed_location
abfss://unity-catalog-storage@<dbstorages>.dfs.core.windows.net/<1234567890>
#### focus_vendor_name
JAMES CONSTRUCTION GROUP, LLC

In [0]:
%python
# Databricks notebook source

# ============================================================
# 00_deploy_objects.py
#
# Purpose:
#   Bootstrap the PHC TXDOT Unity Catalog objects:
#     - validate secret access
#     - create catalog
#     - create schemas
#     - create config / audit tables
#     - create bronze raw closed bids table
#     - apply table and column comments
#
# Source reference for bronze raw table:
#   https://data.texas.gov/dataset/Bid-Tabulations/de7b-7dna/about_data
#
# Notes:
#   - In this workspace, catalog creation requires MANAGED LOCATION.
#   - Pass catalog_managed_location as a widget value when running.
#   - Bronze preserves raw source values as strings.
# ============================================================

# ------------------------------------------------------------
# Widgets
# ------------------------------------------------------------
dbutils.widgets.text("catalog", "phc_txdot")
dbutils.widgets.text("catalog_managed_location", "")
dbutils.widgets.text("secret_scope", "phc-txdot-dev")
dbutils.widgets.text("secret_key", "txdot-app-token")
dbutils.widgets.text("closed_bids_url", "https://data.texas.gov/resource/de7b-7dna.json")
dbutils.widgets.text("closed_bids_source", "closed_bids")
dbutils.widgets.text("focus_vendor_name", "")
dbutils.widgets.text("page_size", "50000")
dbutils.widgets.text("sleep_seconds", "0.5")
dbutils.widgets.text("max_retries", "5")
dbutils.widgets.text("retry_backoff", "1.0")
dbutils.widgets.text("request_timeout", "120")
dbutils.widgets.text("test_mode", "false")
dbutils.widgets.text("test_max_pages", "2")

# ------------------------------------------------------------
# Read widget values
# ------------------------------------------------------------
catalog = dbutils.widgets.get("catalog").strip()
catalog_managed_location = dbutils.widgets.get("catalog_managed_location").strip()
secret_scope = dbutils.widgets.get("secret_scope").strip()
secret_key = dbutils.widgets.get("secret_key").strip()
closed_bids_url = dbutils.widgets.get("closed_bids_url").strip()
closed_bids_source = dbutils.widgets.get("closed_bids_source").strip()
focus_vendor_name = dbutils.widgets.get("focus_vendor_name").strip()
page_size = int(dbutils.widgets.get("page_size"))
sleep_seconds = float(dbutils.widgets.get("sleep_seconds"))
max_retries = int(dbutils.widgets.get("max_retries"))
retry_backoff = float(dbutils.widgets.get("retry_backoff"))
request_timeout = int(dbutils.widgets.get("request_timeout"))
test_mode = dbutils.widgets.get("test_mode").strip().lower() in ("1", "true", "yes", "y")
test_max_pages = int(dbutils.widgets.get("test_max_pages"))

# ------------------------------------------------------------
# Validate required parameters
# ------------------------------------------------------------
required_text_values = {
    "catalog": catalog,
    "catalog_managed_location": catalog_managed_location,
    "secret_scope": secret_scope,
    "secret_key": secret_key,
    "closed_bids_url": closed_bids_url,
    "closed_bids_source": closed_bids_source,
}

missing = [k for k, v in required_text_values.items() if not v]
if missing:
    raise ValueError(
        "Missing required widget values: "
        + ", ".join(missing)
        + ". In this workspace, catalog_managed_location is required."
    )

# ------------------------------------------------------------
# Validate secret access
# ------------------------------------------------------------
_ = dbutils.secrets.get(scope=secret_scope, key=secret_key)

# ------------------------------------------------------------
# Create catalog and schemas
# ------------------------------------------------------------
spark.sql(f"""
CREATE CATALOG IF NOT EXISTS {catalog}
MANAGED LOCATION '{catalog_managed_location}'
""")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.staging")

# ------------------------------------------------------------
# Create staging.pipeline_config
# ------------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.staging.pipeline_config (
      config_key         STRING
    , config_value       STRING
    , updated_at_utc     TIMESTAMP
)
USING DELTA
""")

config_rows = [
    ("closed_bids_url", closed_bids_url),
    ("closed_bids_source", closed_bids_source),
    ("focus_vendor_name", focus_vendor_name),
    ("page_size", str(page_size)),
    ("sleep_seconds", str(sleep_seconds)),
    ("max_retries", str(max_retries)),
    ("retry_backoff", str(retry_backoff)),
    ("request_timeout", str(request_timeout)),
    ("test_mode", str(test_mode).lower()),
    ("test_max_pages", str(test_max_pages)),
    ("secret_scope", secret_scope),
    ("secret_key", secret_key),
    ("catalog_managed_location", catalog_managed_location),
]

config_df = spark.createDataFrame(config_rows, ["config_key", "config_value"])
config_df.createOrReplaceTempView("config_updates")

spark.sql(f"""
MERGE INTO {catalog}.staging.pipeline_config t
USING (
    SELECT
          config_key
        , config_value
        , current_timestamp() AS updated_at_utc
    FROM config_updates
) s
ON t.config_key = s.config_key
WHEN MATCHED THEN UPDATE SET
      t.config_value   = s.config_value
    , t.updated_at_utc = s.updated_at_utc
WHEN NOT MATCHED THEN
    INSERT (
          config_key
        , config_value
        , updated_at_utc
    )
    VALUES (
          s.config_key
        , s.config_value
        , s.updated_at_utc
    )
""")

# ------------------------------------------------------------
# Create staging.audit_watermark
# ------------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.staging.audit_watermark (
      source_name         STRING
    , watermark_value     STRING
    , updated_at_utc      TIMESTAMP
)
USING DELTA
""")

# ------------------------------------------------------------
# Create staging.pipeline_run_log
# ------------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.staging.pipeline_run_log (
      run_id              STRING
    , script_name         STRING
    , status              STRING
    , row_count           BIGINT
    , message             STRING
    , logged_at_utc       TIMESTAMP
)
USING DELTA
""")

# ------------------------------------------------------------
# Create bronze.closed_bids_raw
#
# Source metadata mapping:
#   :id         -> socrata_id
#   :created_at -> socrata_created_at
#   :updated_at -> socrata_updated_at
#   :version    -> socrata_version
# ------------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.bronze.closed_bids_raw (
      socrata_id                          STRING
    , socrata_created_at                  STRING
    , socrata_updated_at                  STRING
    , socrata_version                     STRING
    , bid_code                            STRING
    , bid_item_description                STRING
    , bid_item_quantity                   STRING
    , bid_item_sequence_number            STRING
    , bid_item_unit_price_amount          STRING
    , bid_rank_sequence_number            STRING
    , bid_submit_sequence_number          STRING
    , bid_total_amount                    STRING
    , control_section_job_csj             STRING
    , controlling_project_id_ccsj         STRING
    , county                              STRING
    , dbe_goal_percent                    STRING
    , district_division                   STRING
    , engineer_s_estimate_unit            STRING
    , highway                             STRING
    , low_bidder_flag                     STRING
    , maximum_number_of_working           STRING
    , measurement_unit                    STRING
    , project_actual_let_date             STRING
    , project_classification              STRING
    , project_estimated_let_date          STRING
    , project_id                          STRING
    , project_limits_from                 STRING
    , project_limits_to                   STRING
    , project_name                        STRING
    , project_number                      STRING
    , project_type                        STRING
    , sealed_engineer_s_estimate          STRING
    , sealed_engineer_s_estimate_1        STRING
    , short_description                   STRING
    , spec_book_year                      STRING
    , specification_code                  STRING
    , state_project_number                STRING
    , specification_description           STRING
    , vendor_name                         STRING
    , federal_project_number              STRING
    , force_account_amount                STRING
    , force_account_description           STRING
    , nbi_number                          STRING
    , utility_id                          STRING
    , alternative_bid_code                STRING
    , a_b_engineer_s_estimate_amount      STRING
    , ingestion_run_id                    STRING
    , ingested_at_utc                     STRING
)
USING DELTA
""")

# ------------------------------------------------------------
# Table comments
# ------------------------------------------------------------
spark.sql(f"""
COMMENT ON TABLE {catalog}.bronze.closed_bids_raw IS
'Raw bronze landing table for TxDOT Bid Tabulations sourced from the Texas Open Data Portal.
Reference: https://data.texas.gov/dataset/Bid-Tabulations/de7b-7dna/about_data
This table preserves source values as strings for raw ingestion before silver-layer typing and standardization.'
""")

spark.sql(f"""
COMMENT ON TABLE {catalog}.staging.pipeline_config IS
'Configuration values used by the PHC TXDOT Databricks ingestion and transformation pipeline.'
""")

spark.sql(f"""
COMMENT ON TABLE {catalog}.staging.audit_watermark IS
'Incremental-load watermark tracking by source_name for the PHC TXDOT pipeline.'
""")

spark.sql(f"""
COMMENT ON TABLE {catalog}.staging.pipeline_run_log IS
'Execution log for PHC TXDOT pipeline runs, including status, row counts, and messages.'
""")

# ------------------------------------------------------------
# Column comments: bronze.closed_bids_raw
# ------------------------------------------------------------
column_comments = {
    "socrata_id": "Source row identifier from Socrata metadata (:id).",
    "socrata_created_at": "Source row creation timestamp from Socrata metadata (:created_at).",
    "socrata_updated_at": "Source row last-updated timestamp from Socrata metadata (:updated_at).",
    "socrata_version": "Source row version from Socrata metadata (:version).",
    "bid_code": "Bid code from the source dataset.",
    "bid_item_description": "Bid item description from the source dataset.",
    "bid_item_quantity": "Bid item quantity from the source dataset.",
    "bid_item_sequence_number": "Bid item sequence number from the source dataset.",
    "bid_item_unit_price_amount": "Bid item unit price amount from the source dataset.",
    "bid_rank_sequence_number": "Bid rank sequence number from the source dataset.",
    "bid_submit_sequence_number": "Bid submit sequence number from the source dataset.",
    "bid_total_amount": "Bid total amount from the source dataset.",
    "control_section_job_csj": "Control-section-job identifier from the source dataset.",
    "controlling_project_id_ccsj": "Controlling project identifier from the source dataset.",
    "county": "County from the source dataset.",
    "dbe_goal_percent": "DBE goal percent from the source dataset.",
    "district_division": "District or division from the source dataset.",
    "engineer_s_estimate_unit": "Engineer’s estimate unit from the source dataset.",
    "highway": "Highway from the source dataset.",
    "low_bidder_flag": "Low bidder indicator from the source dataset.",
    "maximum_number_of_working": "Maximum number of working days or working value from the source dataset.",
    "measurement_unit": "Measurement unit from the source dataset.",
    "project_actual_let_date": "Actual project letting date from the source dataset.",
    "project_classification": "Project classification from the source dataset.",
    "project_estimated_let_date": "Estimated project letting date from the source dataset.",
    "project_id": "Project identifier from the source dataset.",
    "project_limits_from": "Project limits from-value from the source dataset.",
    "project_limits_to": "Project limits to-value from the source dataset.",
    "project_name": "Project name from the source dataset.",
    "project_number": "Project number from the source dataset.",
    "project_type": "Project type from the source dataset.",
    "sealed_engineer_s_estimate": "Sealed engineer’s estimate value from the source dataset.",
    "sealed_engineer_s_estimate_1": "Additional sealed engineer’s estimate value from the source dataset.",
    "short_description": "Short description from the source dataset.",
    "spec_book_year": "Specification book year from the source dataset.",
    "specification_code": "Specification code from the source dataset.",
    "state_project_number": "State project number from the source dataset.",
    "specification_description": "Specification description from the source dataset.",
    "vendor_name": "Vendor name from the source dataset.",
    "federal_project_number": "Federal project number from the source dataset.",
    "force_account_amount": "Force account amount from the source dataset.",
    "force_account_description": "Force account description from the source dataset.",
    "nbi_number": "NBI number from the source dataset.",
    "utility_id": "Utility identifier from the source dataset.",
    "alternative_bid_code": "Alternative bid code from the source dataset.",
    "a_b_engineer_s_estimate_amount": "A/B engineer’s estimate amount from the source dataset.",
    "ingestion_run_id": "Pipeline-generated ingestion run identifier for the load that wrote the row.",
    "ingested_at_utc": "UTC timestamp when the row was ingested into the bronze table."
}

for col_name, col_comment in column_comments.items():
    escaped_comment = col_comment.replace("'", "''")
    spark.sql(
        f"COMMENT ON COLUMN {catalog}.bronze.closed_bids_raw.{col_name} IS '{escaped_comment}'"
    )

# ------------------------------------------------------------
# Column comments: staging.pipeline_config
# ------------------------------------------------------------
spark.sql(f"COMMENT ON COLUMN {catalog}.staging.pipeline_config.config_key IS 'Configuration key name.'")
spark.sql(f"COMMENT ON COLUMN {catalog}.staging.pipeline_config.config_value IS 'Configuration value stored as text.'")
spark.sql(f"COMMENT ON COLUMN {catalog}.staging.pipeline_config.updated_at_utc IS 'UTC timestamp when the configuration key was last updated.'")

# ------------------------------------------------------------
# Column comments: staging.audit_watermark
# ------------------------------------------------------------
spark.sql(f"COMMENT ON COLUMN {catalog}.staging.audit_watermark.source_name IS 'Pipeline source name used for incremental watermark tracking.'")
spark.sql(f"COMMENT ON COLUMN {catalog}.staging.audit_watermark.watermark_value IS 'Current incremental watermark value stored for the source.'")
spark.sql(f"COMMENT ON COLUMN {catalog}.staging.audit_watermark.updated_at_utc IS 'UTC timestamp when the watermark was last updated.'")

# ------------------------------------------------------------
# Column comments: staging.pipeline_run_log
# ------------------------------------------------------------
spark.sql(f"COMMENT ON COLUMN {catalog}.staging.pipeline_run_log.run_id IS 'Unique identifier for a pipeline execution run.'")
spark.sql(f"COMMENT ON COLUMN {catalog}.staging.pipeline_run_log.script_name IS 'Notebook or script name that logged the run.'")
spark.sql(f"COMMENT ON COLUMN {catalog}.staging.pipeline_run_log.status IS 'Execution status recorded for the run, such as SUCCESS or FAILED.'")
spark.sql(f"COMMENT ON COLUMN {catalog}.staging.pipeline_run_log.row_count IS 'Number of rows processed or affected during the run.'")
spark.sql(f"COMMENT ON COLUMN {catalog}.staging.pipeline_run_log.message IS 'Execution message associated with the run.'")
spark.sql(f"COMMENT ON COLUMN {catalog}.staging.pipeline_run_log.logged_at_utc IS 'UTC timestamp when the run log entry was written.'")

# ------------------------------------------------------------
# Summary output
# ------------------------------------------------------------
print("=== DEPLOYMENT COMPLETE ===")
print(f"Catalog                 : {catalog}")
print(f"Managed Location        : {catalog_managed_location}")
print(f"Secret Scope            : {secret_scope}")
print(f"Secret Key              : {secret_key}")
print(f"Closed Bids URL         : {closed_bids_url}")
print(f"Closed Bids Source      : {closed_bids_source}")
print(f"Focus Vendor Name       : {focus_vendor_name}")
print(f"Page Size               : {page_size:,}")
print(f"Sleep Seconds           : {sleep_seconds}")
print(f"Max Retries             : {max_retries}")
print(f"Retry Backoff           : {retry_backoff}")
print(f"Request Timeout         : {request_timeout}")
print(f"Test Mode               : {test_mode}")
print(f"Test Max Pages          : {test_max_pages}")
print()
print("Schemas created:")
print(f"  - {catalog}.bronze")
print(f"  - {catalog}.silver")
print(f"  - {catalog}.gold")
print(f"  - {catalog}.staging")
print()
print("Tables created/validated:")
print(f"  - {catalog}.staging.pipeline_config")
print(f"  - {catalog}.staging.audit_watermark")
print(f"  - {catalog}.staging.pipeline_run_log")
print(f"  - {catalog}.bronze.closed_bids_raw")